In [1]:
from torch.utils.data import DataLoader, TensorDataset, random_split
from sagemaker.pytorch import PyTorch

import sagemaker
import logging
import torch
import boto3
import time
import os

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

# s3 = boto3.client('s3')
# s3.list_buckets()

from metrics import eval_model
from train import GenerateModel

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:18                                                                                   │
│                                                                                                  │
│   15 # s3.list_buckets()                                                                         │
│   16                                                                                             │
│   17 from metrics import eval_model                                                              │
│ ❱ 18 from train import GenerateModel                                                             │
│   19                                                                                             │
│                                                                                                  │
│ /home/sagemaker-user/train.py:4 in <module>                                                      │
│                                                                                                  │
│     1 import subprocess                                                                          │
│     2 # subprocess.check_call(["pip", "install", "gensim"])                                      │
│     3 from torch.utils.data import DataLoader, TensorDataset, random_split                       │
│ ❱   4 from gensim.models    import Word2Vec                                                      │
│     5                                                                                            │
│     6 import torch.nn.functional as F                                                            │
│     7 import torch.nn as nn                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'gensim'

In [2]:
model = GenerateModel("Model_Param/processed_full.w2v")
model.eval()
...

Ellipsis

In [3]:
X_test = torch.load("Data/X_test.pt")
Y_test = torch.load("Data/Y_test.pt")

In [4]:
test_loader = DataLoader(TensorDataset(X_test,Y_test),batch_size=64,shuffle=False,worker_init_fn=2)

In [5]:
eval_model(model       = model,
           data_loader = test_loader,
           device      = torch.device("cpu"))

{'acc_macro': 0.0667301959521219,
 'prec_macro': 0.08324095025735176,
 'rec_macro': 0.5216147463819838,
 'f1_macro': 0.14357046614037017,
 'acc_micro': 0.10099930741070545,
 'prec_micro': 0.11301536690137727,
 'rec_micro': 0.4871623556361554,
 'f1_micro': 0.1834684304175129,
 'rec_at_8': 0.1441683166570895,
 'prec_at_8': 0.10287738577212262,
 'f1_at_8': 0.12007219217341986,
 'auc_macro': 0.4836969862805108,
 'auc_micro': 0.4629747659520073}

In [ ]:
# log_curr_path = os.path.join(os.getcwd(),'logger.log')
# logging.basicConfig(filename = log_curr_path,
#                     format   = '%(asctime)s;%(message)s',
#                     filemode = "w",
#                     level    = logging.INFO)
# logger = logging.getLogger()
# logger.info("Some Random Message")

In [2]:
estimator = PyTorch(
    entry_point = "train.py",
    output_path = "s3://sagemaker-us-east-1-762568382963/Results/",
    role        = role,
    py_version  = "py39",
    framework_version="1.13.1",
    instance_count  = 1,
    instance_type   = "ml.g5.xlarge",
    hyperparameters = {"lr"         : 0.01, 
                       "filters"    : 10,
                       "window"     : 3,
                       "num_epochs" : 2,
                       "rounds"     : 10,
                       "batch_size" : 32,
                       "s3_path"    : 'sagemaker-us-east-1-762568382963'}
     )

In [3]:
start = time.time()
estimator.fit()
end   = time.time()
print(f'Runtime: {end - start:,.2f}s')

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2025-09-07-20-30-40-255
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 start = time.time()                                                                          │
│ ❱ 2 estimator.fit()                                                                              │
│   3 end   = time.time()                                                                          │
│   4 print(f'Runtime: {end - start:,.2f}s')                                                       │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:167 in wrapper  │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:138 in wrapper  │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/estimator

In [ ]:
"""
Training seconds: 140
Billable seconds: 140
Runtime: 197.90s
"""

---